In [1]:
import nltk
import re
import random
from nltk.corpus import movie_reviews
from nltk.corpus import stopwords

# Download required NLTK datasets
nltk.download('movie_reviews')
nltk.download('stopwords')
nltk.download('punkt')

# Load the reviews and their corresponding categories (pos/neg)
# We structure this as a list of tuples: (word_list, category)
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

# Shuffle the documents to ensure our training/testing split is randomized
random.shuffle(documents)

print(f"Total reviews loaded: {len(documents)}")
print(f"Sample review words (first 10): {documents[0][0][:10]}")
print(f"Sentiment: {documents[0][1]}")

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Total reviews loaded: 2000
Sample review words (first 10): ['arriving', 'in', 'a', 'barrage', 'of', 'hype', ',', 'the', 'blair', 'witch']
Sentiment: neg


In [2]:
# Create a set of English stop words for O(1) lookup
stop_words = set(stopwords.words('english'))

def clean_text(word_list):
    """
    Cleans a list of words by removing punctuation, normalizing case,
    and filtering out stop words.
    """
    cleaned_words = []
    for word in word_list:
        # Convert to lowercase
        word = word.lower()

        # Keep only alphabetic characters (removes punctuation and numbers)
        if re.match(r'^[a-z]+$', word):
            # Filter out stop words
            if word not in stop_words:
                cleaned_words.append(word)

    return cleaned_words

# Apply the cleaning function to all our documents
cleaned_documents = [(clean_text(words), category) for words, category in documents]

print("--- Before Cleaning ---")
print(documents[0][0][:20])
print("\n--- After Cleaning ---")
print(cleaned_documents[0][0][:20])

--- Before Cleaning ---
['arriving', 'in', 'a', 'barrage', 'of', 'hype', ',', 'the', 'blair', 'witch', 'project', 'is', 'one', 'of', 'the', 'biggest', 'box', 'office', 'success', 'of']

--- After Cleaning ---
['arriving', 'barrage', 'hype', 'blair', 'witch', 'project', 'one', 'biggest', 'box', 'office', 'success', 'year', 'however', 'like', 'golden', 'child', 'although', 'blair', 'witch', 'made']


In [3]:
# Aggregate all words from our cleaned documents
all_words = []
for words, category in cleaned_documents:
    all_words.extend(words)

# Calculate the frequency distribution of all words
freq_dist = nltk.FreqDist(all_words)

# Define our feature vocabulary using the top 3000 most common words
word_features = list(freq_dist.keys())[:3000]

def extract_features(document_words):
    """
    Creates a boolean feature vector for a document.
    Returns a dictionary mapping 'contains(word)' to True/False.
    """
    # Convert document words to a set for O(1) lookup speed
    document_words_set = set(document_words)
    features = {}

    for word in word_features:
        # Check if the vocabulary word exists in this specific document
        features[f'contains({word})'] = (word in document_words_set)

    return features

# Map the feature extraction function across all documents
# This may take a few seconds to run
featuresets = [(extract_features(doc_words), category) for doc_words, category in cleaned_documents]

print(f"Feature extraction complete.")
print(f"Total feature sets: {len(featuresets)}")

Feature extraction complete.
Total feature sets: 2000


In [4]:
# Split the dataset: 1,500 reviews for training, 500 for testing
train_set = featuresets[:1500]
test_set = featuresets[1500:]

# Train the classifier using Bayes' Theorem
classifier = nltk.NaiveBayesClassifier.train(train_set)

# Calculate and print the accuracy on unseen data
accuracy = nltk.classify.accuracy(classifier, test_set)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

# Display the most statistically significant words
print("Top 10 Most Informative Features:")
classifier.show_most_informative_features(10)

Model Accuracy: 78.40%

Top 10 Most Informative Features:
Most Informative Features
         contains(damon) = True              pos : neg    =     13.9 : 1.0
      contains(chilling) = True              pos : neg    =     10.0 : 1.0
        contains(hudson) = True              neg : pos    =      8.6 : 1.0
       contains(passage) = True              pos : neg    =      7.4 : 1.0
      contains(patterns) = True              pos : neg    =      7.4 : 1.0
     contains(illogical) = True              neg : pos    =      7.2 : 1.0
       contains(idiotic) = True              neg : pos    =      7.2 : 1.0
        contains(finest) = True              pos : neg    =      7.2 : 1.0
          contains(gump) = True              pos : neg    =      6.8 : 1.0
       contains(matches) = True              pos : neg    =      6.8 : 1.0


In [6]:
import nltk

nltk.download('punkt_tab') # Added to download the missing resource

def predict_sentiment(review_text):
    """
    Takes a raw string, passes it through the preprocessing pipeline,
    and returns the predicted sentiment.
    """
    # 1. Tokenize the string into a list of words
    words = nltk.word_tokenize(review_text)

    # 2. Clean the words (lowercase, remove punctuation and stopwords)
    cleaned_words = clean_text(words)

    # 3. Convert to the boolean feature dictionary
    features = extract_features(cleaned_words)

    # 4. Ask the Naive Bayes model to predict the class
    prediction = classifier.classify(features)

    return prediction

# Let's test it with a few challenging custom reviews!
review_1 = "The cinematography was absolutely breathtaking, a true visual masterpiece."
review_2 = "I wanted to like it, but the plot was a complete mess and the acting was incredibly wooden."
review_3 = "It wasn't terrible, but it definitely wasn't good either. Very mediocre."

print(f"Review 1 Prediction: {predict_sentiment(review_1)}")
print(f"Review 2 Prediction: {predict_sentiment(review_2)}")
print(f"Review 3 Prediction: {predict_sentiment(review_3)}")

Review 1 Prediction: neg
Review 2 Prediction: neg
Review 3 Prediction: neg


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [7]:
import pickle

# Save the trained classifier to a file
with open('naive_bayes_sentiment_model.pkl', 'wb') as model_file:
    pickle.dump(classifier, model_file)

# Save your word features (vocabulary) so you can reconstruct the exact same feature vectors later
with open('word_features.pkl', 'wb') as features_file:
    pickle.dump(word_features, features_file)

print("Model and vocabulary successfully saved!")

# --- How to load it later ---
# with open('naive_bayes_sentiment_model.pkl', 'rb') as model_file:
#     loaded_classifier = pickle.load(model_file)

Model and vocabulary successfully saved!


## Resources

### Sentiment Analysis Project

#### Project Overview
This project focuses on building a simple sentiment classifier using NLTK to determine whether movie reviews are positive or negative. The goal is to preprocess textual data, extract relevant features, train a machine learning model, and evaluate its performance.

#### Data Source
The dataset used is the `movie_reviews` corpus from the NLTK library, which contains 2000 movie reviews categorized as either positive or negative.

#### Key Steps Performed:
*   **Data Loading and Initial Exploration:** Loaded movie reviews from the NLTK corpus and performed an initial inspection.
*   **Data Preprocessing:** Implemented a `clean_text` function to convert words to lowercase, remove punctuation, and filter out English stop words. This was applied to all review documents.
*   **Feature Extraction:** Utilized NLTK's `FreqDist` to identify the most common words and created a vocabulary of the top 3000 words. A boolean feature extractor was then used to represent each review based on the presence of these vocabulary words.
*   **Model Training & Evaluation (Naive Bayes Classifier):** The dataset was split into training (1500 reviews) and testing (500 reviews sets). A Naive Bayes Classifier was trained on the training set and evaluated on the test set.
*   **Model Performance Visualization:** A confusion matrix heatmap was generated to visually represent the model's performance, showing true positives, true negatives, false positives, and false negatives.
*   **Sentiment Prediction Function:** A utility function `predict_sentiment` was created to preprocess raw text and provide sentiment predictions using the trained model.
*   **Model Saving:** The trained Naive Bayes classifier and the `word_features` vocabulary were saved using `pickle` for future use or deployment.

#### Models and Performance (Accuracy on Test Set):
*   **Naive Bayes Classifier:** Achieved an accuracy of **78.40%** on the test set.

This notebook demonstrates a complete end-to-end natural language processing workflow for sentiment analysis, from data preparation to model training, evaluation, and saving.